In [1]:
%load_ext autoreload
%autoreload 2

from paper_utils import *
    
import os
# os.chdir('../..')
from sklearn.metrics import cohen_kappa_score


# Print entailment inputs for human rating

In [ ]:
runs = {
    '1r6kekju': 'squad-deberta',
    'f8yk94fy': 'trivia_qa-deberta',
    'lzlhybwt': 'bioasq-deberta',
}

configs = {}
for wandb_id in runs:
    configs[wandb_id] = restore_file(wandb_id, filenames=['config.yaml'])[0]
    
check_first_item(configs)

```
contradiction -> 0
neutral -> 1
entailment -> 2
```

In [286]:
def print4human(wandbid, MAX_NO=100, SKIP_UNTIL=0):

    name = runs[wandbid]
    model = '-'.join(name.split('-')[1:])
    slurmid = api.run(f'goatml/semantic_uncertainty/{wandbid}').notes.split(': ')[1]
    last_question = ''
    entailment_counter = 0
    entailment_preds = {}
    skip = False
    
    
    with open(f'../../log/slurm-{slurmid}.out', 'r') as f:
        lines = f.readlines()
        for i, line in enumerate(lines):
            if 'INFO' in line:
                line = line[line.index('INFO') + len('INFO') + 4:]
    
            if f'{model} input:'.lower() in line.lower():
    
                original_line = line
    
                if model == 'deberta':
                    if 'weight room' in line:
                        line = 'How many weight rooms are in the Malkin Athletic Center'
                    elif '?' in line:
                        line = line[:line.index('?')]
                    elif '.' in line[:50]:
                        line = line[:line.index('.')]
                    else:
                        line = line[:40]
                
                if line == last_question:
                    skip = True
                    continue
                else:
                    print(entailment_counter, original_line)
                    skip = False
    
                last_question = line
                entailment_counter += 1
            
            sys.stdout.flush()
            if entailment_counter == MAX_NO:
                break


In [283]:
# squad
print4human('1r6kekju')

0  Deberta Input: What was Warsaw's population in 1901? According to the census of 1901, Warsaw's population was 756,475. -> What was Warsaw's population in 1901? According to the census of 1901, Warsaw's population was 817,000.

1  Deberta Input: When did O2 begin to acculturate in the atmosphere? O2 began to accumulate in the atmosphere approximately 2.7 billion years ago during the Great Oxygenation Event. -> When did O2 begin to acculturate in the atmosphere? Oxygen began to accumulate in the atmosphere around 2.7 billion years ago during the Great Oxygenation Event.

2  Deberta Input: Who was Frédéric Chopin? Frédéric Chopin was a Polish-French Romantic composer and pianist known for his delicate, expressive, and technically demanding piano music. -> Who was Frédéric Chopin? Frédéric Chopin was a Polish composer and pianist of the Romantic era who is widely regarded as one of the greatest composers of all time, known for his technically demanding and expressive piano music, includ

In [287]:
# trivia-qa
print4human('f8yk94fy')

0  Deberta Input: Nigel Hawthorne was Oscar nominated for The Madness of which King? Nigel Hawthorne was Oscar nominated for The Madness of King George. -> Nigel Hawthorne was Oscar nominated for The Madness of which King? Nigel Hawthorne was Oscar-nominated for his portrayal of King George III in The Madness of King George.

1  Deberta Input: Which actress and singer's biography was entitled 'The Other Side Of The Rainbow'? Judy Garland's biography was entitled 'The Other Side Of The Rainbow'. -> Which actress and singer's biography was entitled 'The Other Side Of The Rainbow'? The actress and singer whose biography was entitled 'The Other Side Of The Rainbow' was Judy Garland.

2  Deberta Input: The actor John Wayne was known by what nickname? John Wayne was known by the nickname "The Duke." -> The actor John Wayne was known by what nickname? The actor John Wayne was known by the nickname "The Duke."

3  Deberta Input: In Greek mythology, who did flute playing shepherd Marsyas challe

In [288]:
# bioasq
print4human('lzlhybwt')

0  Deberta Input: Computational tools for predicting allosteric pathways in proteins Computational tools for predicting allosteric pathways in proteins include molecular dynamics simulations, molecular docking, machine learning algorithms, and structure-based design methods. -> Computational tools for predicting allosteric pathways in proteins Computational tools for predicting allosteric pathways in proteins include molecular dynamics simulations, structure-based modeling, machine learning algorithms, and graph theory-based methods, which can aid in identifying potential allosteric sites, predicting the binding of small molecules or protein ligands, and understanding the underlying mechanisms of allostery.

1  Deberta Input: Which is the molecular mechanism underlying K-ras alterations in carcinomas? K-ras alterations in carcinomas involve mutations in the KRAS gene, which can result in the production of a constitutively active K-ras protein that promotes cell proliferation and surviv

# Extract Entailment for existing runs from logs

In [354]:
runs = {
    # '8ehbkd99': 'squad-gpt-4',
    # '5bqecpaq': 'squad-gpt-3.5',
    # 'eriayh79': 'squad-llama-2-70b-chat',
    # '1r6kekju': 'squad-deberta',

    '37it7nr9': 'trivia_qa-gpt-4',
    'mpgj9mz3': 'trivia_qa-gpt-3.5',
    'qfwl6vze': 'trivia_qa-llama-2-70b-chat',
    'f8yk94fy': 'trivia_qa-deberta',
    
    # '3d6obogq': 'bioasq-gpt4',
    # 'jj5da8bd': 'bioasq-gpt3.5',
    # 'lzlhybwt': 'bioasq-deberta',
    # '8i0520i0': 'bioasq-llama-2-70b-chat',
}

In [ ]:
{v: get_slurmid(k) for k, v in runs.items()}

In [332]:
configs = {}
for wandb_id in runs:
    configs[wandb_id] = restore_file(wandb_id, filenames=['config.yaml'])[0]
    
check_first_item(configs)

37it7nr9 167218 trivia_qa  Nigel Hawthorne was Oscar nominated for The Madness of which King?
mpgj9mz3 168315 trivia_qa  Nigel Hawthorne was Oscar nominated for The Madness of which King?
qfwl6vze 171336 trivia_qa  Nigel Hawthorne was Oscar nominated for The Madness of which King?
f8yk94fy 171337 trivia_qa  Nigel Hawthorne was Oscar nominated for The Madness of which King?


In [336]:
get_slurmid = lambda wandbid: api.run(f'goatml/semantic_uncertainty/{wandbid}').notes.split(': ')[1]

In [365]:
# First check runs individually! Then check if they match up!
# Individually: check all 100 inputs make sense!

# CHECKING: does first item line up?
# SKIP_UNTIL = 0
# MAX_NO = 1
# verbose = True

# SKIP_UNTIL = 0
# MAX_NO = 10
# verbose = True

# SKIP_UNTIL = 3
# MAX_NO = 4
# verbose = True

# SKIP_UNTIL = 99
# MAX_NO = 100
# verbose = True


# CHECKING: do things continue to line up at the very end?
# MAX_NO = 99
# SKIP_UNTIL = 98
# verbose = True

MAX_NO = 100
SKIP_UNTIL = 0
verbose = False

# verbose = False


def get_pred(line):
    if 'neutral' in line:
        return 1
    elif 'entailment' in line:
        return 2
    elif 'contradiction' in line:
        return 0
    elif 'prediction: 0' in line:
        return 0
    elif 'prediction: 1' in line:
        return 1
    elif 'prediction: 2' in line:
        return 2
    else:
        raise

preds = {}
for wandbid, name in runs.items():
    # if wandbid != 'f8yk94fy':
        # continue
    
    print(20 * 'xxx')
    print(wandbid, name)

    model = '-'.join(name.split('-')[1:])
    slurmid = get_slurmid(wandbid)
    last_question = ''
    entailment_counter = 0
    entailment_preds = {}
    skip = False
    hit = False
    with open(f'../../log/slurm-{slurmid}.out', 'r') as f:
        lines = f.readlines()
        for i, line in enumerate(lines):
            line = line.lower()

            if 'info' in line:
                line = line[line.index('info'):]

            if f'{model} input:' in line:
                original_line = line

                if model == 'deberta':
                    if 'weight room' in line:
                        line = 'How many weight rooms are in the Malkin Athletic Center'
                    elif '?' in line:
                        line = line[:line.index('?')]
                    elif '.' in line[:50]:
                        line = line[:line.index('.')]
                    else:
                        line = line[:40]
                
                if line == last_question:
                    skip = True
                    hit = False
                    continue
                else:
                    skip = False

                last_question = line
                if verbose and (entailment_counter >= SKIP_UNTIL):
                    print('----')
                    if model == 'deberta':
                        print(i, 'INPT', original_line)
                    else:
                        print(i, 'INPT', line)
                        print(i+2, 'INPT', lines[i+2])
                        print(i+3, 'INPT', lines[i+3])
                entailment_counter += 1

            if not skip and (f'{model} prediction:' in line):
                hit = True
                entailment_prediction = get_pred(line)
                if verbose and (entailment_counter > SKIP_UNTIL):
                    print(i, 'ANSWER', line)
                    print(i, 'ANSWER', entailment_prediction)
                entailment_preds[entailment_counter] = entailment_prediction
            
            sys.stdout.flush()
            if (entailment_counter == MAX_NO) and hit:
                break
    preds[wandbid] = entailment_preds

xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
37it7nr9 trivia_qa-gpt-4
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
mpgj9mz3 trivia_qa-gpt-3.5
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
qfwl6vze trivia_qa-llama-2-70b-chat
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
f8yk94fy trivia_qa-deberta


In [367]:
df = pd.DataFrame.from_dict(preds).rename(columns=lambda x: runs[x])
# for rater in ['squad-human_jansen', 'squad-human_sebhar']:
for rater in [f'{k}-human_jansen' for k in ['trivia_qa', 'bioasq']]:
    df[rater] = 'FILL_IN'
df


,trivia_qa-gpt-4,trivia_qa-gpt-3.5,trivia_qa-llama-2-70b-chat,trivia_qa-deberta,trivia_qa-human_jansen,bioasq-human_jansen
1,2,2,2,1,FILL_IN,FILL_IN
2,2,2,2,2,FILL_IN,FILL_IN
3,2,2,2,2,FILL_IN,FILL_IN
4,2,2,2,1,FILL_IN,FILL_IN
5,2,2,2,2,FILL_IN,FILL_IN
...,...,...,...,...,...,...
96,2,2,2,2,FILL_IN,FILL_IN
97,2,2,2,2,FILL_IN,FILL_IN
98,2,2,2,2,FILL_IN,FILL_IN
99,2,2,2,2,FILL_IN,FILL_IN


In [368]:
tmp = df.reset_index().melt(id_vars='index', value_vars=df.columns)
tmp['dataset'] = tmp.variable.map(lambda x: x.split('-')[0])
tmp['model'] = tmp.variable.map(lambda x: '-'.join(x.split('-')[1:]))
tmp = tmp[['index', 'dataset', 'model', 'value']]
tmp['index'] = tmp['index'] - 1
tmp
# tmp.to_csv('23-11-24-entailment-evaluation.csv', index=False)

,index,dataset,model,value
0,0,trivia_qa,gpt-4,2
1,1,trivia_qa,gpt-4,2
2,2,trivia_qa,gpt-4,2
3,3,trivia_qa,gpt-4,2
4,4,trivia_qa,gpt-4,2
...,...,...,...,...
595,95,bioasq,human_jansen,FILL_IN
596,96,bioasq,human_jansen,FILL_IN
597,97,bioasq,human_jansen,FILL_IN
598,98,bioasq,human_jansen,FILL_IN


In [370]:
print(tmp.to_csv())

,index,dataset,model,value
0,0,trivia_qa,gpt-4,2
1,1,trivia_qa,gpt-4,2
2,2,trivia_qa,gpt-4,2
3,3,trivia_qa,gpt-4,2
4,4,trivia_qa,gpt-4,2
5,5,trivia_qa,gpt-4,2
6,6,trivia_qa,gpt-4,2
7,7,trivia_qa,gpt-4,2
8,8,trivia_qa,gpt-4,2
9,9,trivia_qa,gpt-4,2
10,10,trivia_qa,gpt-4,2
11,11,trivia_qa,gpt-4,2
12,12,trivia_qa,gpt-4,0
13,13,trivia_qa,gpt-4,2
14,14,trivia_qa,gpt-4,2
15,15,trivia_qa,gpt-4,2
16,16,trivia_qa,gpt-4,1
17,17,trivia_qa,gpt-4,2
18,18,trivia_qa,gpt-4,2
19,19,trivia_qa,gpt-4,2
20,20,trivia_qa,gpt-4,2
21,21,trivia_qa,gpt-4,2
22,22,trivia_qa,gpt-4,0
23,23,trivia_qa,gpt-4,2
24,24,trivia_qa,gpt-4,2
25,25,trivia_qa,gpt-4,2
26,26,trivia_qa,gpt-4,2
27,27,trivia_qa,gpt-4,2
28,28,trivia_qa,gpt-4,2
29,29,trivia_qa,gpt-4,1
30,30,trivia_qa,gpt-4,2
31,31,trivia_qa,gpt-4,2
32,32,trivia_qa,gpt-4,2
33,33,trivia_qa,gpt-4,2
34,34,trivia_qa,gpt-4,0
35,35,trivia_qa,gpt-4,2
36,36,trivia_qa,gpt-4,2
37,37,trivia_qa,gpt-4,2
38,38,trivia_qa,gpt-4,2
39,39,trivia_qa,gpt-4,2
40,40,trivia_qa,gpt-4,0
41,41,tri

In [291]:
tmp.model.unique()

array(['gpt-4', 'gpt-3.5', 'llama-2-70b-chat', 'deberta'], dtype=object)

In [289]:
tmp[tmp.model == 'sebhar']

,index,dataset,model,value


# Analysis: Correlation between entailment metrics and human judgement

In [316]:
df = pd.read_csv('23-11-28-entailment-evaluation.csv', index_col=None)

In [317]:
def remove_incomplete(df):
    # filter out incomplete data!
    ignore = []
    for metric, mdf in df.groupby('model'):
        if (mdf.value == 'FILL_IN').any():
            ignore.append(metric)
    print(f'Ignoring metrics {ignore} for now.')
    df = df[df.model.map(lambda x: x not in ignore)]
    return df

df = remove_incomplete(df)

Ignoring metrics [] for now.


In [318]:
for name, tmp in df.groupby(['index', 'dataset', 'model']):
    if len(tmp) > 1:
        print(name)
        display(tmp)

In [328]:
def agreement(x, y):
    return np.mean(x == y)


for method in ['pearson', 'kendall', 'spearman', agreement, cohen_kappa_score]:
    print(90*'x')
    print(colorize(f'METHOD: {method}'))
    print(90*'x')

    for dataset, gdf in df.groupby('dataset'):
        pdf = gdf.pivot(index='index', columns='model', values='value')    
        print(colorize(f'method: {method} -- dataset: {dataset}', 1))
        display(pdf.corr(method=method))

xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
METHOD: pearson
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
method: pearson -- dataset: bioasq


model,human_sebhar
model,
human_sebhar,1.0


method: pearson -- dataset: squad


model,deberta,gpt-3.5,gpt-4,human_jansen,human_sebhar,llama-2-70b-chat
model,,,,,,
deberta,1.000000,0.694685,0.730558,0.620574,0.633727,0.512429
gpt-3.5,0.694685,1.000000,0.740407,0.690731,0.650650,0.528964
gpt-4,0.730558,0.740407,1.000000,0.748663,0.616427,0.447512
human_jansen,0.620574,0.690731,0.748663,1.000000,0.675886,0.462562
human_sebhar,0.633727,0.650650,0.616427,0.675886,1.000000,0.604424
llama-2-70b-chat,0.512429,0.528964,0.447512,0.462562,0.604424,1.000000


method: pearson -- dataset: trivia_qa


...

model,human_sebhar
model,
human_sebhar,1.0


method: <function cohen_kappa_score at 0x7fed6421d940> -- dataset: squad


model,deberta,gpt-3.5,gpt-4,human_jansen,human_sebhar,llama-2-70b-chat
model,,,,,,
deberta,1.000000,0.384846,0.532992,0.475734,0.390606,0.252951
gpt-3.5,0.384846,1.000000,0.613713,0.611964,0.354940,0.391368
gpt-4,0.532992,0.613713,1.000000,0.640849,0.311254,0.316781
human_jansen,0.475734,0.611964,0.640849,1.000000,0.428934,0.334384
human_sebhar,0.390606,0.354940,0.311254,0.428934,1.000000,0.350877
llama-2-70b-chat,0.252951,0.391368,0.316781,0.334384,0.350877,1.000000


method: <function cohen_kappa_score at 0x7fed6421d940> -- dataset: trivia_qa


model,human_sebhar
model,
human_sebhar,1.0
